# TinyTimeMixer Forecasting

In [ ]:
import pandas as pd
import torch
from transformers import TinyTimeMixerForPrediction
import matplotlib.pyplot as plt

## Load Data

In [ ]:
df_hist = pd.read_parquet('../../data/hist.parquet')
df_future = pd.read_parquet('../../data/future_covariates.parquet')

# Ensure the timestamp is the index
df_hist['time_idx'] = pd.to_datetime(df_hist['time_idx'])
df_hist = df_hist.set_index('time_idx')

print("Historical Data:")
print(df_hist.head())
print("\nFuture Covariates:")
print(df_future.head())

## Prepare Data for Model

In [ ]:
prediction_length = 12 # Predict 12 steps into the future
context_length = 64   # Use 64 past steps for context

# Select a single time series to forecast (e.g., item_id = 'T1')
target_series = df_hist[df_hist['item_id'] == 'T1']['value'].values

# Get the context window (the last `context_length` points)
context = target_series[-context_length:]

# Convert to tensor
past_values = torch.tensor(context, dtype=torch.float32).unsqueeze(0)


## Load Model and Predict

In [ ]:
# Load pre-trained model
model = TinyTimeMixerForPrediction.from_pretrained("ibm/TTM")

# Generate forecast
with torch.no_grad():
    outputs = model.generate(
        past_values=past_values,
        prediction_length=prediction_length
    )
    forecast = outputs.sequences.numpy().squeeze()

## Visualize Results

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(target_series[-context_length:], label='Historical Data')
forecast_index = pd.to_datetime(df_hist[df_hist['item_id'] == 'T1'].index[-1]) + pd.to_timedelta(range(1, prediction_length + 1), unit='H')
plt.plot(forecast_index, forecast[-prediction_length:], label='Forecast', color='red')
plt.title('TinyTimeMixer Forecast vs. Historical Data')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()